# Notebook 04 — HCHO Hotspots & Fire Correlation

**Objective 2:** Detect HCHO hotspots related to biomass burning.

Steps demonstrated:

1. Load HCHO + fire gridded dataset (or generate synthetic)
2. Seasonal mean HCHO maps over India
3. 90th / 95th percentile hotspot identification
4. DBSCAN clustering of hotspot regions
5. Time-lag cross-correlation between HCHO and fire counts
6. Wind vector overlay on hotspot maps

> Synthetic data is generated if `data/processed/hcho_fire_daily_grid.csv` is absent.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.cluster import DBSCAN

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Libraries loaded ✓')

## 1. Load HCHO + Fire Dataset

Columns: `date, lat, lon, hcho_column, fire_count, u10, v10, season`.
HCHO is in mol m⁻² (TROPOMI); fire_count is daily FIRMS pixel count per grid cell.

In [ ]:
data_path = Path('../data/processed/hcho_fire_daily_grid.csv')

SEASON_MAP = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Pre-monsoon', 4: 'Pre-monsoon', 5: 'Pre-monsoon',
    6: 'Monsoon', 7: 'Monsoon', 8: 'Monsoon', 9: 'Monsoon',
    10: 'Post-monsoon', 11: 'Post-monsoon',
}

if data_path.exists():
    df = pd.read_csv(data_path, parse_dates=['date'])
    print(f'✓ Loaded real HCHO+Fire data: {df.shape}')
else:
    print('Data not found — generating synthetic HCHO + fire grid…')
    rng = np.random.default_rng(3)
    lats = np.arange(8.0, 37.5, 0.5)
    lons = np.arange(68.0, 97.5, 0.5)
    dates = pd.date_range('2022-01-01', '2022-12-31', freq='7D')
    records = []
    for d in dates:
        m = d.month
        for lat in lats:
            for lon in lons:
                igp  = (23 <= lat <= 30) and (75 <= lon <= 90)
                ne   = (23 <= lat <= 28) and (90 <= lon <= 97.5)
                ph   = (28 <= lat <= 32) and (73 <= lon <= 78)  # Punjab-Haryana
                # HCHO higher in post-monsoon (burning) and monsoon (vegetation)
                hcho_base = 1.5e-4
                hcho = hcho_base * rng.lognormal(0, 0.5)
                if m in (10, 11) and (ph or igp):
                    hcho *= rng.uniform(2.5, 4.0)
                elif m in (6, 7, 8, 9):
                    hcho *= rng.uniform(1.3, 2.0)
                # Fire count
                fire_base = 0
                if m in (10, 11) and ph:
                    fire_base = rng.poisson(8)
                elif m in (2, 3, 4) and (ne or igp):
                    fire_base = rng.poisson(4)
                fire_count = fire_base + rng.poisson(0.3)
                records.append({
                    'date': d, 'lat': round(lat, 1), 'lon': round(lon, 1),
                    'hcho_column': round(float(hcho), 8),
                    'fire_count':  int(fire_count),
                    'u10': float(rng.normal(2, 3)),
                    'v10': float(rng.normal(0, 2)),
                })
    df = pd.DataFrame(records)
    print(f'✓ Synthetic grid: {df.shape}')

df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['season'] = df['month'].map(SEASON_MAP)
print(f'Date range: {df["date"].min().date()} – {df["date"].max().date()}')
df.head(3)

## 2. Seasonal Mean HCHO Maps

Post-monsoon (Oct–Nov) typically shows the highest HCHO in Punjab-Haryana
(paddy residue burning) and northeast India.
Monsoon has elevated biogenic HCHO from vegetation.

In [ ]:
seasons = ['Winter', 'Pre-monsoon', 'Monsoon', 'Post-monsoon']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, season in zip(axes, seasons):
    sub = df[df['season'] == season]
    if sub.empty:
        ax.set_title(f'{season} (no data)')
        continue
    piv = sub.pivot_table(index='lat', columns='lon',
                           values='hcho_column', aggfunc='mean')
    vmax = df.groupby('season')['hcho_column'].mean().max() * 1.5
    im = ax.pcolormesh(piv.columns, piv.index, piv.values,
                       cmap='YlOrRd', vmin=0, vmax=vmax, shading='auto')
    plt.colorbar(im, ax=ax, label='HCHO (mol m⁻²)', shrink=0.85, pad=0.02)
    ax.set_xlim(68, 97.5); ax.set_ylim(8, 37.5)
    ax.set_xlabel('°E'); ax.set_ylabel('°N')
    ax.set_title(season, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.2)

fig.suptitle('Seasonal Mean HCHO Column — Sentinel-5P TROPOMI', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Hotspot Identification — 90th/95th Percentile

Cells exceeding the 90th seasonal percentile of HCHO are flagged as hotspots.
Post-monsoon hotspots in Punjab-Haryana are well-known biomass burning signals.

In [ ]:
# Compute seasonal percentile thresholds
pct90 = df.groupby('season')['hcho_column'].quantile(0.90)
pct95 = df.groupby('season')['hcho_column'].quantile(0.95)
print('90th percentile HCHO by season:')
print(pct90.to_string())

# Flag hotspots per row
df['threshold_90'] = df['season'].map(pct90)
df['threshold_95'] = df['season'].map(pct95)
df['is_hotspot_90'] = df['hcho_column'] >= df['threshold_90']
df['is_hotspot_95'] = df['hcho_column'] >= df['threshold_95']

hotspot_rate = df.groupby('season')['is_hotspot_90'].mean() * 100
print('\nHotspot rate (%) by season (90th pct):')
print(hotspot_rate.round(1).to_string())

# Plot post-monsoon hotspot map
pm_df = df[df['season'] == 'Post-monsoon']
if pm_df.empty:
    pm_df = df[df['month'].isin([10, 11])]

if not pm_df.empty:
    fig, ax = plt.subplots(figsize=(9, 7))
    piv_h = pm_df.pivot_table(index='lat', columns='lon',
                               values='hcho_column', aggfunc='mean')
    im = ax.pcolormesh(piv_h.columns, piv_h.index, piv_h.values,
                       cmap='YlOrRd', shading='auto')
    plt.colorbar(im, ax=ax, label='Mean HCHO (mol m⁻²)', shrink=0.85)

    # Overlay hotspot cells
    hs = pm_df[pm_df['is_hotspot_90']].groupby(['lat','lon']).size().reset_index(name='n')
    ax.scatter(hs['lon'], hs['lat'], c='red', s=6, alpha=0.6, label='Hotspot (90th pct)')

    ax.set_xlim(68, 97.5); ax.set_ylim(8, 37.5)
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_title('Post-Monsoon HCHO Hotspots (Oct–Nov)', fontweight='bold')
    ax.legend(markerscale=3, fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. DBSCAN Clustering of Hotspot Regions

DBSCAN groups spatially contiguous hotspot cells into labelled regions.
Known clusters: Punjab-Haryana (paddy burning), Northeast India (slash-and-burn),
Central India (forest fires).

In [ ]:
pm_hot = (
    df[(df['season'] == 'Post-monsoon') & df['is_hotspot_90']]
    .groupby(['lat','lon'])['hcho_column'].mean()
    .reset_index()
)

if len(pm_hot) < 5:
    pm_hot = df[df['is_hotspot_90']].groupby(['lat','lon'])['hcho_column'].mean().reset_index()

coords = pm_hot[['lat','lon']].values

db = DBSCAN(eps=1.5, min_samples=4, metric='euclidean').fit(coords)
pm_hot['cluster'] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise    = (db.labels_ == -1).sum()
print(f'DBSCAN found {n_clusters} clusters | {n_noise} noise points')

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.get_cmap('tab10', max(1, n_clusters))

noise_pts = pm_hot[pm_hot['cluster'] == -1]
ax.scatter(noise_pts['lon'], noise_pts['lat'], c='lightgray', s=6, alpha=0.5,
           label='Noise')

for cid in range(n_clusters):
    cluster_pts = pm_hot[pm_hot['cluster'] == cid]
    ax.scatter(cluster_pts['lon'], cluster_pts['lat'],
               c=[cmap(cid)], s=15, alpha=0.8, label=f'Cluster {cid}')
    cx, cy = cluster_pts['lon'].mean(), cluster_pts['lat'].mean()
    ax.text(cx, cy + 0.3, f'C{cid}', fontsize=8, fontweight='bold',
            color=cmap(cid), ha='center')

ax.set_xlim(68, 97.5); ax.set_ylim(8, 37.5)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title(f'DBSCAN Hotspot Clusters (Post-monsoon) — {n_clusters} clusters',
             fontweight='bold')
ax.legend(fontsize=8, loc='lower right', ncol=2)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Time-Lag Correlation: HCHO vs Fire Count

After biomass burning, HCHO can persist for several days in the atmosphere
(photochemical lifetime ~1–2 days).  We compute Pearson-r at lags 0–3 days
for each region defined in `config/hcho_hotspot.yaml`.

In [ ]:
REGIONS = {
    'Punjab-Haryana':       {'lat': (28, 32), 'lon': (73, 78)},
    'Indo-Gangetic Plain':  {'lat': (23, 30), 'lon': (75, 90)},
    'Northeast India':      {'lat': (23, 28), 'lon': (90, 97.5)},
    'Central India Forests':{'lat': (18, 25), 'lon': (77, 87)},
}

LAG_DAYS = [0, 1, 2, 3]

all_cors = {}
for region_name, bbox in REGIONS.items():
    mask = (
        (df['lat'] >= bbox['lat'][0]) & (df['lat'] <= bbox['lat'][1]) &
        (df['lon'] >= bbox['lon'][0]) & (df['lon'] <= bbox['lon'][1])
    )
    reg_ts = (
        df[mask].groupby('date')[['hcho_column','fire_count']]
        .mean().sort_index()
    )
    if len(reg_ts) < 10:
        all_cors[region_name] = {lag: np.nan for lag in LAG_DAYS}
        continue
    cors = {}
    for lag in LAG_DAYS:
        hcho_s  = reg_ts['hcho_column'].iloc[lag:].values
        fire_s  = reg_ts['fire_count'].iloc[:len(hcho_s)].values
        valid   = ~(np.isnan(hcho_s) | np.isnan(fire_s))
        if valid.sum() > 5:
            r, p = pearsonr(hcho_s[valid], fire_s[valid])
            cors[lag] = round(r, 3)
        else:
            cors[lag] = np.nan
    all_cors[region_name] = cors

cors_df = pd.DataFrame(all_cors, index=LAG_DAYS).T
cors_df.columns = [f'Lag {l}d' for l in LAG_DAYS]
print('Pearson r (HCHO vs fire count) at different lags:')
print(cors_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
markers = ['o', 's', '^', 'D']
colors_r = ['#d32f2f','#1565c0','#2e7d32','#f57c00']

for (region, row), marker, color in zip(cors_df.iterrows(), markers, colors_r):
    vals = row.values.astype(float)
    ax.plot(LAG_DAYS, vals, marker=marker, color=color, linewidth=2,
            markersize=8, label=region)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(0.3, color='gray', linewidth=0.6, linestyle=':', alpha=0.6)
ax.set_xlabel('Lag (days)  —  HCHO lags fire by N days')
ax.set_ylabel('Pearson r')
ax.set_title('HCHO–Fire Cross-Correlation at Lag 0–3 days', fontweight='bold')
ax.set_xticks(LAG_DAYS)
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.show()

## 6. Wind Transport Overlay on Hotspot Map

ERA5 10-m wind vectors are overlaid on the post-monsoon HCHO field.
Arrows indicate the dominant transport direction from burning regions.

In [ ]:
pm_avg = (
    df[df['season'] == 'Post-monsoon']
    .groupby(['lat','lon'])
    .agg(hcho=('hcho_column','mean'), u10=('u10','mean'), v10=('v10','mean'))
    .reset_index()
)
if pm_avg.empty:
    pm_avg = (
        df[df['month'].isin([10, 11])]
        .groupby(['lat','lon'])
        .agg(hcho=('hcho_column','mean'), u10=('u10','mean'), v10=('v10','mean'))
        .reset_index()
    )

piv_hcho = pm_avg.pivot(index='lat', columns='lon', values='hcho')
piv_u    = pm_avg.pivot(index='lat', columns='lon', values='u10')
piv_v    = pm_avg.pivot(index='lat', columns='lon', values='v10')

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.pcolormesh(piv_hcho.columns, piv_hcho.index, piv_hcho.values,
                   cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='Mean HCHO (mol m⁻²)', shrink=0.85)

# Sub-sample wind vectors for readability
skip = max(1, len(piv_hcho.columns) // 18)
lons_q = piv_u.columns[::skip]
lats_q = piv_u.index[::skip]
u_q = piv_u.reindex(index=lats_q, columns=lons_q)
v_q = piv_v.reindex(index=lats_q, columns=lons_q)
LON_Q, LAT_Q = np.meshgrid(lons_q, lats_q)
ax.quiver(LON_Q, LAT_Q, u_q.values, v_q.values,
          scale=150, width=0.004, color='navy', alpha=0.75)

# Highlight key regions
from matplotlib.patches import Rectangle
for rname, bbox, c_rect in [
    ('Punjab-Haryana', {'lat':(28,32),'lon':(73,78)}, 'red'),
    ('Northeast',      {'lat':(23,28),'lon':(90,97.5)}, 'blue'),
]:
    rect = Rectangle(
        (bbox['lon'][0], bbox['lat'][0]),
        bbox['lon'][1]-bbox['lon'][0], bbox['lat'][1]-bbox['lat'][0],
        linewidth=2, edgecolor=c_rect, facecolor='none',
    )
    ax.add_patch(rect)
    ax.text(bbox['lon'][0]+0.2, bbox['lat'][0]+0.2, rname,
            color=c_rect, fontsize=8, fontweight='bold')

ax.set_xlim(68, 97.5); ax.set_ylim(8, 37.5)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title(
    'Post-Monsoon HCHO with ERA5 Wind Vectors\n'
    '(arrows show 10-m wind transport direction)',
    fontweight='bold',
)
ax.grid(True, linestyle='--', alpha=0.25)
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('- Red box: Punjab-Haryana paddy residue burning (Oct–Nov)')
print('- Blue box: Northeast India slash-and-burn (Mar–Apr)')
print('- Arrows: prevailing westerly flow transports HCHO downwind into IGP')